# 04. Abstractive Summarization with Transformers (BART & T5)
## Text Summarization Using Pre-trained Models

**Project:** Academic NLP Summarization Pipeline  
**Objective:** Implement sequence-to-sequence abstractive text generation using pre-trained `facebook/bart-large-cnn` and `google-t5/t5-small`, utilizing PyTorch CUDA FP16 mixed precision on an RTX 3060 Laptop GPU (6GB VRAM).

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd

from src.preprocessing import load_processed_split
from src.summarizer import get_summarizer

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

## 1. Load Preprocessed Test Sample
We load a real news article from `data/processed/processed_test_sample.csv`.

In [ ]:
test_df = load_processed_split("test")
sample = test_df.iloc[0]

print(f"Article ID: {sample['id']}")
print("\n--- Human Reference Highlights ---")
print(sample['highlights'])

## 2. BART-Large-CNN Abstractive Summarization
BART (Lewis et al., 2020) is a bidirectional encoder and autoregressive decoder pretrained by corrupting text and reconstructing original documents. The `facebook/bart-large-cnn` model was fine-tuned specifically on CNN/DailyMail.

In [ ]:
bart = get_summarizer("bart")
bart_result = bart.summarize_text(sample['article'])

print("--- BART Generated Summary ---")
print(bart_result['summary'])
print(f"\nLatency:      {bart_result['latency_sec']}s")
print(f"Input Tokens: {bart_result['input_tokens']}")
print(f"Output Tokens:{bart_result['output_tokens']}")
print(f"Device:       {bart_result['device']}")

## 3. T5-Small Abstractive Summarization
T5 (Raffel et al., 2020) is an encoder-decoder model framing all NLP tasks as text-to-text problems. For summarization, the input is prefixed with `summarize: `.

In [ ]:
t5 = get_summarizer("t5")
t5_result = t5.summarize_text(sample['article'])

print("--- T5 Generated Summary ---")
print(t5_result['summary'])
print(f"\nLatency:      {t5_result['latency_sec']}s")
print(f"Input Tokens: {t5_result['input_tokens']}")
print(f"Output Tokens:{t5_result['output_tokens']}")

## 4. Side-by-Side Qualitative Comparison
Notice how BART constructs natural, syntactically fluent sentences that closely capture the key factual essence of the story.

In [ ]:
pd.DataFrame({
    "Model": ["Human Reference", "BART Large CNN", "T5 Small"],
    "Generated Summary": [sample['highlights'], bart_result['summary'], t5_result['summary']],
    "Latency (s)": [0.0, bart_result['latency_sec'], t5_result['latency_sec']],
    "Word Count": [len(sample['highlights'].split()), len(bart_result['summary'].split()), len(t5_result['summary'].split())]
})